# Notebook 13 - Speech Recognition and Transcription

This notebook covers the practical transcription baseline for multimodal systems: segment audio, transcribe it, and keep timestamps for later grounding.


## What you will learn

- how to think about speech transcription as a grounding problem
- how Whisper-style pipelines package audio into segments
- how to evaluate transcript quality with simple metrics


In [ ]:
!pip install -q "transformers>=4.57.0" accelerate torch numpy pandas matplotlib librosa soundfile torchaudio
print("Installed notebook dependencies.")


In [ ]:
import json
import math
import random
import statistics
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-multimodal")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_ROOT = ARTIFACT_ROOT / "13_speech_recognition_and_transcription"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(7)
np.random.seed(7)



print("Cache dir:", CACHE_DIR)
print("Artifact root:", NOTEBOOK_ROOT.resolve())


## Step 1: Create a toy waveform

A synthetic waveform is enough to make sampling rate and time axis ideas concrete.


In [ ]:
sample_rate = 16000
seconds = 2.0
time_axis = np.linspace(0, seconds, int(sample_rate * seconds), endpoint=False)
waveform = 0.4 * np.sin(2 * np.pi * 440 * time_axis) + 0.2 * np.sin(2 * np.pi * 660 * time_axis)
plt.figure(figsize=(10, 3))
plt.plot(time_axis[:2000], waveform[:2000])
plt.title("Toy waveform excerpt")
plt.xlabel("seconds")
plt.show()


## Step 2: Sketch a Whisper-style transcription call

The heavy model call is not the interesting part. The interface and the artifact shape are.


In [ ]:
whisper_example = {
    "model": "openai/whisper-small",
    "input_audio": "meeting_clip.wav",
    "output": {
        "segments": [
            {"start": 0.0, "end": 4.8, "text": "Let's review the rollout issues first."},
            {"start": 4.8, "end": 9.7, "text": "The login flow failed after reset."},
        ]
    },
}
print(json.dumps(whisper_example, indent=2))


## Step 3: Measure transcript error

Word error rate is simple enough to teach and still useful as a baseline.


In [ ]:
def word_error_rate(reference, hypothesis):
    ref = reference.lower().split()
    hyp = hypothesis.lower().split()
    dp = [[0] * (len(hyp) + 1) for _ in range(len(ref) + 1)]
    for i in range(len(ref) + 1):
        dp[i][0] = i
    for j in range(len(hyp) + 1):
        dp[0][j] = j
    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return dp[-1][-1] / max(1, len(ref))

reference = "the login flow failed after reset"
hypothesis = "the login flow fail after reset"
print("WER:", round(word_error_rate(reference, hypothesis), 3))


## Exercises and extensions

1. Run the same example with a real Whisper checkpoint and compare the transcript segments.
1. Store transcript segments as evidence spans so later QA answers can cite timestamps.
